# Phase-1 Drive-First Pipeline (Colab)

Goal: run existing Python scripts as-is and persist artifacts directly in Google Drive.

Flow:
`00 -> 08 -> (manual labels check) -> 09 -> 10 -> 11 -> 06 -> 07`


## 0) Mount Drive + Prepare Repository On Drive

- Repository path (default): `/content/drive/MyDrive/UNLV-Research`
- `Phase-1` outputs are written under Drive path, so runtime disconnect does not erase files.


In [ ]:
import os
import sys
import csv
import json
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/LeeKeonSoo/UNLV-Research.git'
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_REPO_DIR = Path(os.environ.get('PHASE1_DRIVE_REPO_DIR', str(DRIVE_ROOT / 'UNLV-Research'))).expanduser()
PHASE1_DIR = DRIVE_REPO_DIR / 'Phase-1'

if not (DRIVE_REPO_DIR / '.git').exists():
    DRIVE_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print('[Repo] cloning ->', DRIVE_REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, str(DRIVE_REPO_DIR)], check=True)
else:
    print('[Repo] exists ->', DRIVE_REPO_DIR)
    subprocess.run(['git', '-C', str(DRIVE_REPO_DIR), 'pull'], check=False)

if not PHASE1_DIR.exists():
    raise FileNotFoundError(f'Phase-1 not found at {PHASE1_DIR}')

os.environ['PHASE1_REPO_DIR'] = str(DRIVE_REPO_DIR)
os.environ['PHASE1_PHASE1_DIR'] = str(PHASE1_DIR)
os.environ['PHASE1_AUTOSAVE_ROOT'] = str(PHASE1_DIR)
os.environ['PYTHONUNBUFFERED'] = '1'

PYTHON = sys.executable
print('Python      :', PYTHON)
print('Repo (Drive):', DRIVE_REPO_DIR)
print('Phase-1     :', PHASE1_DIR)


def run_py(script_name: str, *args: str):
    cmd = [PYTHON, '-u', str(PHASE1_DIR / script_name), *args]
    print()
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=str(PHASE1_DIR), text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{script_name} failed with exit code {r.returncode}')


## 1) Optional Dependency Install


In [ ]:
INSTALL_DEPS = False
if INSTALL_DEPS:
    req = PHASE1_DIR / 'requirements.txt'
    if req.exists():
        subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(req)], check=False)
    else:
        print('requirements.txt not found; skipped.')
else:
    print('Dependency install skipped.')


## 2) Fixed Execution Profile


In [ ]:
profile = {
    'PHASE1_DEVICE': 'auto',
    'PHASE1_CUDA_DEVICE': '0',
    'PHASE1_USE_GPU': '1',
    'PHASE1_DOMAIN_BATCH_SIZE': '256',
    'PHASE1_MAX_BATCHES': '0',
    'PHASE1_INDEX_MAX_BATCHES': '0',
    'PHASE1_TFIDF_MAX_FEATURES': '3000',
    'PHASE1_BUILD_TFIDF_MATRIX': '0',
    'PHASE1_STORE_DOC_TEXTS': '1',
    'PHASE1_DOC_TEXT_BACKEND': 'sqlite',
    'PHASE1_DOC_TEXT_INSERT_BATCH': '2000',
    'PHASE1_DOC_TEXT_CACHE_LIMIT': '2500',
    'PHASE1_ENABLE_MINHASH': '1',
    'PHASE1_QUERY_CACHE_LIMIT': '1024',
    'PHASE1_NGRAM_CACHE_LIMIT': '2048',
    'PHASE1_SEMANTIC_CANDIDATE_LIMIT': '80',
    'PHASE1_NGRAM_CANDIDATE_LIMIT': '80',
    'PHASE1_SKIP_REDUNDANCY': '0',
    'PHASE1_SKIP_PERPLEXITY': '0',
    'PHASE1_DATASETS_CONFIG': 'datasets_config.json',
    'PHASE1_IDENTITY_CONFIG': 'configs/metric_identity_v1.json',
    'PHASE1_AUTOSAVE_ROOT': str(PHASE1_DIR),
}

for k, v in profile.items():
    os.environ[k] = str(v)

print('Profile loaded.')


## 3) Run Core + Certification Chain

Behavior:
- If core outputs are missing, runs `00_run_phase1.py`
- If labels are incomplete, runs `08_generate_label_templates.py` then stops for manual labeling
- If labels are complete, continues `09 -> 10 -> 11 -> 06 -> 07`


In [ ]:
core_outputs = [
    PHASE1_DIR / 'outputs' / 'run_manifest.json',
    PHASE1_DIR / 'outputs' / 'khan_analysis.jsonl',
    PHASE1_DIR / 'outputs' / 'tiny_textbooks_analysis.jsonl',
]

if all(p.exists() for p in core_outputs):
    print('[SKIP] 00_run_phase1.py (core outputs already exist)')
else:
    print('[RUN] 00_run_phase1.py (core outputs missing)')
    run_py('00_run_phase1.py')

labels_dir = PHASE1_DIR / 'validation' / 'labels'
checks = {
    'domain_labels_v1.csv': ('gold_primary', 'main_eval'),
    'quality_labels_v1.csv': ('gold_has_examples', 'main_eval'),
    'difficulty_sanity_v1.csv': ('gold_valid', 'main_eval'),
    'ood_labels_v1.csv': ('gold_in_domain', 'calibration'),
}

def non_empty(v):
    return str(v or '').strip() != ''

def label_status():
    missing = []
    detail = []
    for name, (col, split_name) in checks.items():
        p = labels_dir / name
        if not p.exists():
            missing.append(f'{name}: missing file')
            detail.append((name, split_name, 0, 0))
            continue
        with p.open('r', encoding='utf-8', newline='') as f:
            rows = list(csv.DictReader(f))
        split_rows = [r for r in rows if str(r.get('split', '')).strip() == split_name]
        labeled = [r for r in split_rows if non_empty(r.get(col))]
        detail.append((name, split_name, len(split_rows), len(labeled)))
        if len(split_rows) == 0 or len(labeled) == 0:
            missing.append(f'{name}: split={split_name} labeled=0')
    return missing, detail

missing, detail = label_status()
for name, split_name, total, labeled in detail:
    print(f'[LABEL] {name} split={split_name} total={total} labeled={labeled}')

if missing:
    print()
    print('Labels are incomplete. Generating templates via 08...')
    run_py('08_generate_label_templates.py')
    missing, detail = label_status()
    for name, split_name, total, labeled in detail:
        print(f'[LABEL-AFTER-08] {name} split={split_name} total={total} labeled={labeled}')

if missing:
    print()
    print('Manual labeling required before 09:')
    for m in missing:
        print(' -', m)
    raise SystemExit('Stop here. Fill labels in Drive, then rerun this cell.')

run_py('09_calibrate_ood_thresholds.py')
run_py('10_score_metric_gates.py')
run_py('11_certify_phase1.py')
run_py('06_build_dashboard.py')
run_py('07_validate_phase1_outputs.py', '--require-gates', '--write-report', 'outputs/validation/final_validation_report.json')

print()
print('Full chain completed.')


## 4) Output Checkpoint (Drive)


In [ ]:
outs = [
    PHASE1_DIR / 'outputs' / 'run_manifest.json',
    PHASE1_DIR / 'outputs' / 'run_summary.json',
    PHASE1_DIR / 'outputs' / 'dashboard.html',
    PHASE1_DIR / 'outputs' / 'validation' / 'ood_calibration_report.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'gate_scores_main.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'gate_scores_transfer.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'full_validation_report.json',
    PHASE1_DIR / 'outputs' / 'validation' / 'final_validation_report.json',
]

for p in outs:
    print(f"[{'OK' if p.exists() else 'MISS'}] {p}")

manifest_path = PHASE1_DIR / 'outputs' / 'run_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print()
    print('certification_status:', manifest.get('certification_status'))
    print('certification_failed_gates:', manifest.get('certification_failed_gates'))
